# 07 — Blind Manual Correctness Annotation

Bu aşamada ilk kez gerçek target oluşturulur:

```text
manual_is_correct ∈ {0, 1}
```

## Temel metodoloji

Annotator yalnız şunları görür:

- soru
- 04'te dondurulmuş model cevabı
- doğrulanmış reference answer
- accepted aliases
- answer type / category
- varsa verification source

Annotator **şunları görmez**:

- `mean_logprob`
- entropy feature'ları
- self-confidence
- self-verification
- self-consistency
- modelin gerçek kimliği

Model kimliği `M1 / M2 / M3` şeklinde anonimleştirilir ve mapping ayrı internal dosyada tutulur.

Bu, confidence sinyallerinin correctness etiketini bilinçsiz biçimde etkilemesini azaltır.

## Annotation kararı

`manual_is_correct`:

```text
1 = Frozen model answer soruyu olgusal olarak doğru yanıtlıyor
0 = Yanlış / yetersiz / unknown / çelişkili
```

Yanlış cevaplarda `manual_error_type` şu değerlerden biri olmalıdır:

```text
WRONG_FACT
UNKNOWN
MIXED_OR_CONTRADICTORY
INCOMPLETE_OR_TRUNCATED
IRRELEVANT_OR_FORMAT
OTHER
```

Doğru cevaplarda `manual_error_type` boş bırakılabilir; finalize sırasında otomatik `CORRECT`
olarak yazılır.

Kararsız kaldığınız satırda:

```text
needs_review = 1
```

yazabilirsiniz. Stage-07 bu kayıt çözülmeden finalize edilmez.

!!! UYARI -> manual_annotation_work.csv tamamlayınca csv olarak aktar ve manual_annotation_completed.csv ismini ver.
## Dataset version carry-forward

Yeni dataset sürümüne soru eklediğinizde `PREVIOUS_ANNOTATION_DATASET_VERSION` alanına
annotation yaptığınız önceki sürümü yazın. Notebook önceki `manual_annotation_completed.csv`
satırlarını korur ve yalnız yeni/değişmiş satırları boş olarak sonraki batch'lere ekler.


## 1. Drive ve imports

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
import hashlib
import json
import math
import os
import random
import unicodedata

from datetime import datetime, timezone
from pathlib import Path

import pandas as pd

## 2. Project paths

In [3]:
PROJECT_ROOT = Path("/content/drive/MyDrive/LLM_Confidence_Thesis")

DATA_VALIDATED_DIR = PROJECT_ROOT / "data" / "validated"
ANNOTATIONS_ROOT = PROJECT_ROOT / "data" / "annotations"
RESULTS_STAGE04_DIR = PROJECT_ROOT / "results" / "stage04"
RESULTS_STAGE06_DIR = PROJECT_ROOT / "results" / "stage06"
RESULTS_STAGE07_DIR = PROJECT_ROOT / "results" / "stage07"

for p in [
    ANNOTATIONS_ROOT,
    RESULTS_STAGE07_DIR,
]:
    p.mkdir(parents=True, exist_ok=True)

# ============================================================
# CURRENT dataset version
VALIDATED_DATASET_VERSION = "validated_v003"

# Daha önce manual annotation yaptığınız dataset sürümü.
# İlk kez annotation yapıyorsanız None bırakın.
PREVIOUS_ANNOTATION_DATASET_VERSION = "validated_v002"

# True ise mevcut current-version work/internal dosyaları,
# yeni carry-forward yapısıyla uyuşmuyorsa timestamp'li backup alınarak yenilenir.
REBUILD_CURRENT_ANNOTATION_WORK = True
# ============================================================

if (
    PREVIOUS_ANNOTATION_DATASET_VERSION
    ==
    VALIDATED_DATASET_VERSION
):
    raise ValueError(
        "PREVIOUS_ANNOTATION_DATASET_VERSION current version ile aynı olamaz."
    )

DATASET_JSON_PATH = (
    DATA_VALIDATED_DIR /
    f"{VALIDATED_DATASET_VERSION}.json"
)

DATASET_MANIFEST_PATH = (
    DATA_VALIDATED_DIR /
    f"{VALIDATED_DATASET_VERSION}_manifest.json"
)

STAGE04_COMPLETION_PATH = (
    RESULTS_STAGE04_DIR /
    f"{VALIDATED_DATASET_VERSION}_stage04_complete.json"
)

STAGE06_COMPLETION_PATH = (
    RESULTS_STAGE06_DIR /
    f"{VALIDATED_DATASET_VERSION}_stage06_v2_complete.json"
)

ANNOTATION_DIR = (
    ANNOTATIONS_ROOT /
    VALIDATED_DATASET_VERSION
)

ANNOTATION_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

ANNOTATION_TEMPLATE_PATH = (
    ANNOTATION_DIR /
    "manual_annotation_work.csv"
)

ANNOTATION_COMPLETED_PATH = (
    ANNOTATION_DIR /
    "manual_annotation_completed.csv"
)

ANNOTATION_INTERNAL_MAP_PATH = (
    ANNOTATION_DIR /
    "manual_annotation_internal_map.jsonl"
)

MODEL_CODE_MAP_PATH = (
    ANNOTATION_DIR /
    "model_code_map.json"
)

ANNOTATION_PROTOCOL_MANIFEST_PATH = (
    ANNOTATION_DIR /
    "annotation_protocol_manifest.json"
)

ANNOTATION_INSTRUCTIONS_PATH = (
    ANNOTATION_DIR /
    "ANNOTATION_INSTRUCTIONS.md"
)

FINAL_LABELS_PATH = (
    ANNOTATION_DIR /
    "manual_labels_final.jsonl"
)

FINAL_LABELS_CSV_PATH = (
    ANNOTATION_DIR /
    "manual_labels_final.csv"
)

STAGE07_COMPLETION_PATH = (
    RESULTS_STAGE07_DIR /
    f"{VALIDATED_DATASET_VERSION}_stage07_complete.json"
)

# ------------------------------------------------------------
# Optional previous annotation paths
# ------------------------------------------------------------
if PREVIOUS_ANNOTATION_DATASET_VERSION is not None:
    PREVIOUS_ANNOTATION_DIR = (
        ANNOTATIONS_ROOT /
        PREVIOUS_ANNOTATION_DATASET_VERSION
    )

    PREVIOUS_ANNOTATION_COMPLETED_PATH = (
        PREVIOUS_ANNOTATION_DIR /
        "manual_annotation_completed.csv"
    )

    PREVIOUS_ANNOTATION_INTERNAL_MAP_PATH = (
        PREVIOUS_ANNOTATION_DIR /
        "manual_annotation_internal_map.jsonl"
    )

    PREVIOUS_MODEL_CODE_MAP_PATH = (
        PREVIOUS_ANNOTATION_DIR /
        "model_code_map.json"
    )
else:
    PREVIOUS_ANNOTATION_DIR = None
    PREVIOUS_ANNOTATION_COMPLETED_PATH = None
    PREVIOUS_ANNOTATION_INTERNAL_MAP_PATH = None
    PREVIOUS_MODEL_CODE_MAP_PATH = None

print("Current dataset version:", VALIDATED_DATASET_VERSION)
print("Carry annotations from:", PREVIOUS_ANNOTATION_DATASET_VERSION)


## 3. Hash / serialization helpers

In [4]:
def canonical_json(obj):
    return json.dumps(
        obj,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
    )

def sha256_text(text):
    return hashlib.sha256(
        text.encode("utf-8")
    ).hexdigest()

def sha256_object(obj):
    return sha256_text(
        canonical_json(obj)
    )

def file_sha256(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b"",
        ):
            h.update(chunk)

    return h.hexdigest()

def utc_now_iso():
    return datetime.now(
        timezone.utc
    ).isoformat()

def read_jsonl(path):
    rows = []

    if not path.exists():
        return rows

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()

            if line:
                rows.append(
                    json.loads(line)
                )

    return rows

def write_jsonl_exclusive(path, rows):
    with open(
        path,
        "x",
        encoding="utf-8",
    ) as f:
        for row in rows:
            f.write(
                json.dumps(
                    row,
                    ensure_ascii=False,
                    allow_nan=False,
                )
                + "\n"
            )

def stringify_aliases(value):
    if value is None:
        return ""

    if isinstance(
        value,
        list,
    ):
        return " | ".join(
            str(x)
            for x in value
        )

    return str(value)

def stringify_source(value):
    if value is None:
        return ""

    if isinstance(
        value,
        (
            dict,
            list,
        ),
    ):
        return json.dumps(
            value,
            ensure_ascii=False,
            sort_keys=True,
        )

    return str(value)

## 4. Önceki stage ve validated dataset bütünlüğü

In [5]:
for required in [
    DATASET_JSON_PATH,
    DATASET_MANIFEST_PATH,
    STAGE04_COMPLETION_PATH,
]:
    if not required.exists():
        raise FileNotFoundError(
            required
        )

if not STAGE06_COMPLETION_PATH.exists():
    raise FileNotFoundError(
        "Stage-06 V2 completion manifest yok. "
        "Önce 06 V2'nin son completion hücresini çalıştırın:\n"
        f"{STAGE06_COMPLETION_PATH}"
    )

with open(
    DATASET_JSON_PATH,
    "r",
    encoding="utf-8",
) as f:
    validated_data = json.load(f)

with open(
    DATASET_MANIFEST_PATH,
    "r",
    encoding="utf-8",
) as f:
    dataset_manifest = json.load(f)

dataset_sha256 = file_sha256(
    DATASET_JSON_PATH
)

if (
    dataset_sha256
    !=
    dataset_manifest[
        "validated_file_sha256"
    ]
):
    raise RuntimeError(
        "Validated dataset SHA manifest ile uyuşmuyor."
    )

if (
    len(
        validated_data
    )
    !=
    dataset_manifest[
        "n_validated_questions"
    ]
):
    raise RuntimeError(
        "Validated dataset N manifest ile uyuşmuyor."
    )

with open(
    STAGE04_COMPLETION_PATH,
    "r",
    encoding="utf-8",
) as f:
    stage04_manifest = json.load(f)

with open(
    STAGE06_COMPLETION_PATH,
    "r",
    encoding="utf-8",
) as f:
    stage06_manifest = json.load(f)

if (
    stage04_manifest[
        "dataset_sha256"
    ]
    !=
    dataset_sha256
):
    raise RuntimeError(
        "Stage-04 farklı validated dataset SHA taşıyor."
    )

if (
    stage06_manifest[
        "dataset_version"
    ]
    !=
    VALIDATED_DATASET_VERSION
):
    raise RuntimeError(
        "Stage-06 dataset version uyuşmuyor."
    )

print("Validated dataset:", len(validated_data))
print("Stage-04 model-answer rows:", stage04_manifest["n_model_answer_rows"])
print("Stage-06 V2 model-answer rows:", stage06_manifest["n_model_answer_rows"])
print("Integrity: PASS")

Validated dataset: 200
Stage-04 model-answer rows: 600
Stage-06 V2 model-answer rows: 600
Integrity: PASS


## 5. Reference dataset tablosu

In [6]:
reference_df = pd.DataFrame(
    validated_data
)

required_reference_columns = {
    "question_id",
    "question",
    "reference_answer",
    "accepted_aliases",
    "answer_type",
    "category",
}

missing = (
    required_reference_columns
    -
    set(
        reference_df.columns
    )
)

if missing:
    raise RuntimeError(
        f"Validated dataset eksik kolonlar: {sorted(missing)}"
    )

if reference_df[
    "question_id"
].duplicated().any():
    raise RuntimeError(
        "Validated dataset duplicate question_id içeriyor."
    )

reference_lookup = (
    reference_df
    .set_index(
        "question_id"
    )
    .to_dict(
        orient="index"
    )
)

print("Reference lookup ready:", len(reference_lookup))

Reference lookup ready: 200


## 6. Stage-04 frozen cevaplarını yükle

In [7]:
stage04_rows = []

for artifact in (
    stage04_manifest[
        "model_exports"
    ]
):
    path = Path(
        artifact[
            "export_file"
        ]
    )

    if not path.exists():
        raise FileNotFoundError(
            path
        )

    if (
        file_sha256(
            path
        )
        !=
        artifact[
            "export_sha256"
        ]
    ):
        raise RuntimeError(
            f"Stage-04 export SHA mismatch: {artifact['model_id']}"
        )

    rows = read_jsonl(
        path
    )

    for row in rows:
        stage04_rows.append(
            row
        )

stage04_df = pd.DataFrame(
    stage04_rows
)

expected_rows = (
    stage04_manifest[
        "n_questions"
    ]
    *
    stage04_manifest[
        "n_models"
    ]
)

if len(
    stage04_df
) != expected_rows:
    raise RuntimeError(
        f"Beklenen Stage-04 rows={expected_rows}; "
        f"bulunan={len(stage04_df)}"
    )

if stage04_df.duplicated(
    subset=[
        "question_id",
        "model_id",
    ]
).any():
    raise RuntimeError(
        "Duplicate Stage-04 question_id × model_id."
    )

print("Frozen answer rows:", len(stage04_df))

Frozen answer rows: 600


## 7. Blind model-code mapping

Gerçek model kimlikleri annotation CSV'sinde gösterilmez.

`PREVIOUS_ANNOTATION_DATASET_VERSION` verilmişse önceki sürümde kullanılan `M1 / M2 / M3`
eşlemesi korunur. Böylece önceki completed annotation satırları güvenli biçimde current
dataset sürümüne taşınabilir.

Mapping immutable internal JSON olarak tutulur.


In [8]:
ANNOTATION_PROTOCOL_VERSION = (
    "blind_manual_correctness_v2_carryforward"
)

MODEL_CODE_SEED = 7319
ROW_SHUFFLE_SEED = 9143
ANNOTATION_BATCH_SIZE = 120

model_ids = sorted(
    stage04_df[
        "model_id"
    ].unique()
)

previous_model_code_map = {}

if PREVIOUS_ANNOTATION_DATASET_VERSION is not None:
    if not PREVIOUS_MODEL_CODE_MAP_PATH.exists():
        raise FileNotFoundError(
            "Previous model-code mapping bulunamadı:\n"
            f"{PREVIOUS_MODEL_CODE_MAP_PATH}"
        )

    with open(
        PREVIOUS_MODEL_CODE_MAP_PATH,
        "r",
        encoding="utf-8",
    ) as f:
        previous_map_payload = json.load(f)

    previous_model_code_map = (
        previous_map_payload[
            "model_id_to_code"
        ]
    )

# Önceki sürümdeki model code'larını koru.
# Yeni model eklenmişse yalnız yeni modellere yeni code ver.
generated_model_code_map = {}

for model_id in model_ids:
    if model_id in previous_model_code_map:
        generated_model_code_map[
            model_id
        ] = previous_model_code_map[
            model_id
        ]

new_model_ids = [
    model_id
    for model_id in model_ids
    if model_id not in generated_model_code_map
]

rng = random.Random(
    MODEL_CODE_SEED
)

rng.shuffle(
    new_model_ids
)

used_code_numbers = []

for code in generated_model_code_map.values():
    if (
        isinstance(code, str)
        and code.startswith("M")
        and code[1:].isdigit()
    ):
        used_code_numbers.append(
            int(code[1:])
        )

next_code_number = (
    max(used_code_numbers, default=0)
    + 1
)

for model_id in new_model_ids:
    generated_model_code_map[
        model_id
    ] = (
        f"M{next_code_number}"
    )

    next_code_number += 1

if MODEL_CODE_MAP_PATH.exists():
    with open(
        MODEL_CODE_MAP_PATH,
        "r",
        encoding="utf-8",
    ) as f:
        existing_map = json.load(f)

    model_code_map = (
        existing_map[
            "model_id_to_code"
        ]
    )

    if (
        model_code_map
        !=
        generated_model_code_map
    ):
        raise RuntimeError(
            "Existing current-version model-code mapping "
            "carry-forward mapping ile uyuşmuyor.\n"
            "Current annotation klasöründeki model_code_map.json "
            "önceki başarısız denemeden kaldıysa kontrol edin."
        )
else:
    model_code_map = (
        generated_model_code_map
    )

    with open(
        MODEL_CODE_MAP_PATH,
        "x",
        encoding="utf-8",
    ) as f:
        json.dump(
            {
                "protocol_version": (
                    ANNOTATION_PROTOCOL_VERSION
                ),
                "seed": (
                    MODEL_CODE_SEED
                ),
                "inherited_from_dataset_version": (
                    PREVIOUS_ANNOTATION_DATASET_VERSION
                ),
                "model_id_to_code": (
                    model_code_map
                ),
                "created_at_utc": (
                    utc_now_iso()
                ),
            },
            f,
            ensure_ascii=False,
            indent=2,
        )

print(
    "Blind model codes ready:",
    sorted(
        model_code_map.values()
    ),
)

if PREVIOUS_ANNOTATION_DATASET_VERSION is not None:
    inherited_count = sum(
        1
        for model_id
        in model_ids
        if model_id
        in previous_model_code_map
    )

    print(
        "Inherited model-code mappings:",
        inherited_count,
    )

print(
    "Gerçek model mapping annotation sırasında açılmamalıdır."
)


Blind model codes ready: ['M1', 'M2', 'M3']
Gerçek model mapping annotation sırasında açılmamalıdır.


## 8. Annotation work CSV oluştur + previous completed annotation'ları taşı

Akış:

1. Current Stage-04 satırlarından annotation candidate'ları oluşturulur.
2. `PREVIOUS_ANNOTATION_DATASET_VERSION` içindeki `manual_annotation_completed.csv` yüklenir.
3. Değişmemiş eski annotation satırları, label ve sıra/batch bilgileriyle aynen üstte korunur.
4. Yeni veya değişmiş satırlar boş annotation alanlarıyla eski satırların altına eklenir.
5. Yeni satırlar önceki son batch numarasından sonraki batch'lere atanır.

Eski bir satırın frozen model answer/reference/alias/type/category/source bilgisi değişmişse
etiket otomatik taşınmaz; yeniden annotation için boş bırakılır.


In [9]:
EDITABLE_COLUMNS = [
    "manual_is_correct",
    "manual_error_type",
    "annotation_notes",
    "needs_review",
]

VISIBLE_PROTECTED_COLUMNS = [
    "annotation_row_id",
    "annotation_batch",
    "model_code",
    "question_id",
    "question",
    "model_answer",
    "reference_answer",
    "accepted_aliases",
    "answer_type",
    "category",
    "verification_source",
]

PROTECTED_FINGERPRINT_FIELDS = [
    "annotation_row_id",
    "model_code",
    "question_id",
    "question",
    "model_answer",
    "reference_answer",
    "accepted_aliases",
    "answer_type",
    "category",
    "verification_source",
]

ERROR_TYPES_INCORRECT = {
    "WRONG_FACT",
    "UNKNOWN",
    "MIXED_OR_CONTRADICTORY",
    "INCOMPLETE_OR_TRUNCATED",
    "IRRELEVANT_OR_FORMAT",
    "OTHER",
}

VISIBLE_CORE_FIELDS = [
    field
    for field in VISIBLE_PROTECTED_COLUMNS
    if field
    not in {
        "annotation_row_id",
        "annotation_batch",
    }
]

def csv_text(value):
    if value is None:
        return ""

    try:
        if pd.isna(value):
            return ""
    except (TypeError, ValueError):
        pass

    return str(value)

def backup_file(path):
    timestamp = datetime.now(
        timezone.utc
    ).strftime(
        "%Y%m%dT%H%M%SZ"
    )

    backup_path = path.with_name(
        f"{path.stem}.backup_{timestamp}{path.suffix}"
    )

    counter = 1

    while backup_path.exists():
        backup_path = path.with_name(
            f"{path.stem}.backup_{timestamp}_{counter}{path.suffix}"
        )
        counter += 1

    path.replace(
        backup_path
    )

    return backup_path

# ============================================================
# 1. Current dataset için annotation candidate'larını üret
# ============================================================
source_column_candidates = [
    "verification_source",
    "source_reference",
]

current_candidates = []
current_source_lookup = {}

for _, model_row in (
    stage04_df
    .iterrows()
):
    question_id_native = (
        model_row[
            "question_id"
        ]
    )

    model_id = csv_text(
        model_row[
            "model_id"
        ]
    )

    question_id_key = csv_text(
        question_id_native
    )

    current_key = (
        question_id_key,
        model_id,
    )

    if current_key in current_source_lookup:
        raise RuntimeError(
            f"Duplicate current annotation key: {current_key}"
        )

    ref = (
        reference_lookup[
            question_id_native
        ]
    )

    source_value = ""

    for candidate in (
        source_column_candidates
    ):
        if (
            candidate
            in ref
            and
            ref[
                candidate
            ]
            not in [
                None,
                "",
            ]
        ):
            source_value = (
                stringify_source(
                    ref[
                        candidate
                    ]
                )
            )
            break

    visible_core = {
        "model_code": csv_text(
            model_code_map[
                model_id
            ]
        ),
        "question_id": (
            question_id_key
        ),
        "question": csv_text(
            ref[
                "question"
            ]
        ),
        "model_answer": csv_text(
            model_row[
                "answer"
            ]
        ),
        "reference_answer": csv_text(
            ref[
                "reference_answer"
            ]
        ),
        "accepted_aliases": csv_text(
            stringify_aliases(
                ref[
                    "accepted_aliases"
                ]
            )
        ),
        "answer_type": csv_text(
            ref[
                "answer_type"
            ]
        ),
        "category": csv_text(
            ref[
                "category"
            ]
        ),
        "verification_source": csv_text(
            source_value
        ),
    }

    current_candidates.append({
        "_current_key": (
            current_key
        ),
        "_model_id": (
            model_id
        ),
        **visible_core,
    })

    current_source_lookup[
        current_key
    ] = model_row.to_dict()

current_candidate_lookup = {
    row[
        "_current_key"
    ]: row
    for row
    in current_candidates
}

# ============================================================
# 2. Previous completed annotation'ları yükle ve güvenli olanları taşı
# ============================================================
carried_rows = []
carried_keys = set()
row_id_to_current_key = {}
row_id_origin = {}

previous_row_count = 0
previous_labeled_count = 0
dropped_previous_count = 0
changed_previous_count = 0

previous_annotation_df = None
previous_internal_lookup = {}

if PREVIOUS_ANNOTATION_DATASET_VERSION is not None:
    for required_previous_path in [
        PREVIOUS_ANNOTATION_COMPLETED_PATH,
        PREVIOUS_ANNOTATION_INTERNAL_MAP_PATH,
        PREVIOUS_MODEL_CODE_MAP_PATH,
    ]:
        if not required_previous_path.exists():
            raise FileNotFoundError(
                "Previous annotation carry-forward için dosya eksik:\n"
                f"{required_previous_path}"
            )

    previous_annotation_df = pd.read_csv(
        PREVIOUS_ANNOTATION_COMPLETED_PATH,
        dtype=str,
        keep_default_na=False,
    )

    previous_internal_rows = (
        read_jsonl(
            PREVIOUS_ANNOTATION_INTERNAL_MAP_PATH
        )
    )

    required_previous_columns = set(
        VISIBLE_PROTECTED_COLUMNS
        +
        EDITABLE_COLUMNS
    )

    missing_previous_columns = (
        required_previous_columns
        -
        set(
            previous_annotation_df.columns
        )
    )

    if missing_previous_columns:
        raise RuntimeError(
            "Previous completed annotation CSV eksik kolonlar içeriyor: "
            f"{sorted(missing_previous_columns)}"
        )

    if previous_annotation_df[
        "annotation_row_id"
    ].duplicated().any():
        raise RuntimeError(
            "Previous completed annotation duplicate annotation_row_id içeriyor."
        )

    previous_internal_lookup = {
        csv_text(
            row[
                "annotation_row_id"
            ]
        ): row
        for row
        in previous_internal_rows
    }

    if (
        len(
            previous_internal_lookup
        )
        !=
        len(
            previous_internal_rows
        )
    ):
        raise RuntimeError(
            "Previous internal map duplicate annotation_row_id içeriyor."
        )

    previous_row_count = len(
        previous_annotation_df
    )

    previous_labeled_count = int(
        previous_annotation_df[
            "manual_is_correct"
        ]
        .astype(str)
        .str.strip()
        .isin(
            [
                "0",
                "1",
            ]
        )
        .sum()
    )

    # Previous CSV sırasını aynen koru.
    for _, previous_row in (
        previous_annotation_df
        .iterrows()
    ):
        annotation_row_id = csv_text(
            previous_row[
                "annotation_row_id"
            ]
        )

        if (
            annotation_row_id
            not in
            previous_internal_lookup
        ):
            raise RuntimeError(
                "Previous completed CSV'deki row ID "
                "internal map'te bulunamadı: "
                f"{annotation_row_id}"
            )

        previous_internal = (
            previous_internal_lookup[
                annotation_row_id
            ]
        )

        previous_question_id = csv_text(
            previous_internal[
                "question_id"
            ]
        )

        previous_model_id = csv_text(
            previous_internal[
                "model_id"
            ]
        )

        if (
            csv_text(
                previous_row[
                    "question_id"
                ]
            )
            !=
            previous_question_id
        ):
            raise RuntimeError(
                "Previous completed CSV / internal map question_id mismatch: "
                f"{annotation_row_id}"
            )

        current_key = (
            previous_question_id,
            previous_model_id,
        )

        current_candidate = (
            current_candidate_lookup.get(
                current_key
            )
        )

        # Soru/model current dataset'te artık yoksa taşıma.
        if current_candidate is None:
            dropped_previous_count += 1
            continue

        # Annotation item değiştiyse eski etiketi otomatik taşımıyoruz.
        # Özellikle model_answer/reference/source değişimleri yeniden annotation gerektirir.
        compatible = all(
            csv_text(
                previous_row[
                    field
                ]
            )
            ==
            csv_text(
                current_candidate[
                    field
                ]
            )
            for field
            in VISIBLE_CORE_FIELDS
        )

        if not compatible:
            changed_previous_count += 1
            continue

        if current_key in carried_keys:
            raise RuntimeError(
                f"Previous annotation duplicate question×model carry key: {current_key}"
            )

        carried_row = {
            "annotation_row_id": (
                annotation_row_id
            ),
            "annotation_batch": csv_text(
                previous_row[
                    "annotation_batch"
                ]
            ),
            **{
                field: csv_text(
                    current_candidate[
                        field
                    ]
                )
                for field
                in VISIBLE_CORE_FIELDS
            },
            **{
                field: csv_text(
                    previous_row[
                        field
                    ]
                )
                for field
                in EDITABLE_COLUMNS
            },
        }

        carried_rows.append(
            carried_row
        )

        carried_keys.add(
            current_key
        )

        row_id_to_current_key[
            annotation_row_id
        ] = current_key

        row_id_origin[
            annotation_row_id
        ] = (
            PREVIOUS_ANNOTATION_DATASET_VERSION
        )

# ============================================================
# 3. Yalnız current version'da yeniden annotation gereken satırları üret
# ============================================================
new_candidates = [
    row
    for row
    in current_candidates
    if row[
        "_current_key"
    ]
    not in carried_keys
]

new_candidates_df = pd.DataFrame(
    new_candidates
)

if not new_candidates_df.empty:
    new_candidates_df = (
        new_candidates_df
        .sample(
            frac=1.0,
            random_state=ROW_SHUFFLE_SEED,
        )
        .reset_index(
            drop=True
        )
    )

# Yeni ID'ler previous annotation ID'lerinin devamından başlar.
used_annotation_numbers = []

if previous_annotation_df is not None:
    for annotation_row_id in (
        previous_annotation_df[
            "annotation_row_id"
        ]
        .astype(str)
        .tolist()
    ):
        if (
            annotation_row_id.startswith("ANN")
            and
            annotation_row_id[
                3:
            ].isdigit()
        ):
            used_annotation_numbers.append(
                int(
                    annotation_row_id[
                        3:
                    ]
                )
            )

next_annotation_number = (
    max(
        used_annotation_numbers,
        default=0,
    )
    + 1
)

# Yeni batch her zaman previous annotation'ın son batch'inden sonra başlar.
previous_batch_numbers = []

if previous_annotation_df is not None:
    for batch_value in (
        previous_annotation_df[
            "annotation_batch"
        ]
        .astype(str)
        .tolist()
    ):
        if batch_value.strip().isdigit():
            previous_batch_numbers.append(
                int(
                    batch_value.strip()
                )
            )

first_new_batch = (
    max(
        previous_batch_numbers,
        default=0,
    )
    + 1
)

new_rows = []

for new_index, (_, candidate_row) in enumerate(
    new_candidates_df.iterrows()
):
    annotation_row_id = (
        f"ANN{next_annotation_number + new_index:06d}"
    )

    annotation_batch = (
        first_new_batch
        +
        (
            new_index
            //
            ANNOTATION_BATCH_SIZE
        )
    )

    current_key = (
        candidate_row[
            "_current_key"
        ]
    )

    new_row = {
        "annotation_row_id": (
            annotation_row_id
        ),
        "annotation_batch": (
            str(
                annotation_batch
            )
        ),
        **{
            field: csv_text(
                candidate_row[
                    field
                ]
            )
            for field
            in VISIBLE_CORE_FIELDS
        },
        "manual_is_correct": "",
        "manual_error_type": "",
        "annotation_notes": "",
        "needs_review": "0",
    }

    new_rows.append(
        new_row
    )

    row_id_to_current_key[
        annotation_row_id
    ] = current_key

    row_id_origin[
        annotation_row_id
    ] = (
        VALIDATED_DATASET_VERSION
    )

# Previous rows üstte, yeni batch'ler altta.
template_rows = (
    carried_rows
    +
    new_rows
)

template_df = pd.DataFrame(
    template_rows,
    columns=(
        VISIBLE_PROTECTED_COLUMNS
        +
        EDITABLE_COLUMNS
    ),
)

if len(
    template_df
) != len(
    current_candidates
):
    raise RuntimeError(
        "Carry-forward sonrası annotation coverage current Stage-04 ile uyuşmuyor. "
        f"template={len(template_df)}, current={len(current_candidates)}"
    )

if template_df[
    "annotation_row_id"
].duplicated().any():
    raise RuntimeError(
        "Generated template duplicate annotation_row_id içeriyor."
    )

# ============================================================
# 4. Current internal map + protected fingerprint
# ============================================================
internal_rows = []

for _, visible_row in (
    template_df
    .iterrows()
):
    annotation_row_id = csv_text(
        visible_row[
            "annotation_row_id"
        ]
    )

    current_key = (
        row_id_to_current_key[
            annotation_row_id
        ]
    )

    source = (
        current_source_lookup[
            current_key
        ]
    )

    protected_payload = {
        field: csv_text(
            visible_row[
                field
            ]
        )
        for field
        in PROTECTED_FINGERPRINT_FIELDS
    }

    internal_rows.append({
        "annotation_row_id": (
            annotation_row_id
        ),
        "question_id": csv_text(
            visible_row[
                "question_id"
            ]
        ),
        "model_code": csv_text(
            visible_row[
                "model_code"
            ]
        ),
        "model_id": csv_text(
            source[
                "model_id"
            ]
        ),
        "model_revision": csv_text(
            source[
                "model_revision"
            ]
        ),
        "stage04_inference_cache_key": csv_text(
            source[
                "inference_cache_key"
            ]
        ),
        "model_answer_sha256": (
            sha256_text(
                csv_text(
                    source[
                        "answer"
                    ]
                )
            )
        ),
        "annotation_origin_dataset_version": (
            row_id_origin[
                annotation_row_id
            ]
        ),
        "protected_row_fingerprint": (
            sha256_object(
                protected_payload
            )
        ),
    })

# ============================================================
# 5. Work/internal artifact'larını yaz
# ============================================================
write_work_file = True

if ANNOTATION_TEMPLATE_PATH.exists():
    existing_template = pd.read_csv(
        ANNOTATION_TEMPLATE_PATH,
        dtype=str,
        keep_default_na=False,
    )

    desired_template = (
        template_df
        .astype(str)
    )

    same_template = (
        list(
            existing_template.columns
        )
        ==
        list(
            desired_template.columns
        )
        and
        existing_template.equals(
            desired_template
        )
    )

    if same_template:
        write_work_file = False

        print(
            "Annotation work CSV zaten güncel; üzerine yazılmadı."
        )
    else:
        if not REBUILD_CURRENT_ANNOTATION_WORK:
            raise RuntimeError(
                "Mevcut current-version annotation work CSV "
                "yeni carry-forward template ile uyuşmuyor. "
                "REBUILD_CURRENT_ANNOTATION_WORK=True yapın "
                "veya mevcut dosyayı manuel kontrol edin."
            )

        backup_path = backup_file(
            ANNOTATION_TEMPLATE_PATH
        )

        print(
            "Eski annotation work CSV backup alındı:",
            backup_path,
        )

if write_work_file:
    template_df.to_csv(
        ANNOTATION_TEMPLATE_PATH,
        index=False,
        encoding="utf-8-sig",
    )

    print(
        "Annotation work CSV yazıldı:",
        ANNOTATION_TEMPLATE_PATH,
    )

write_internal_file = True

if ANNOTATION_INTERNAL_MAP_PATH.exists():
    existing_internal = (
        read_jsonl(
            ANNOTATION_INTERNAL_MAP_PATH
        )
    )

    if (
        existing_internal
        ==
        internal_rows
    ):
        write_internal_file = False

        print(
            "Internal annotation mapping zaten güncel; üzerine yazılmadı."
        )
    else:
        if not REBUILD_CURRENT_ANNOTATION_WORK:
            raise RuntimeError(
                "Mevcut internal annotation mapping "
                "yeni carry-forward mapping ile uyuşmuyor."
            )

        backup_path = backup_file(
            ANNOTATION_INTERNAL_MAP_PATH
        )

        print(
            "Eski internal map backup alındı:",
            backup_path,
        )

if write_internal_file:
    write_jsonl_exclusive(
        ANNOTATION_INTERNAL_MAP_PATH,
        internal_rows,
    )

    print(
        "Internal mapping yazıldı:",
        ANNOTATION_INTERNAL_MAP_PATH,
    )

# ============================================================
# 6. Carry-forward özeti
# ============================================================
n_carried = len(
    carried_rows
)

n_new = len(
    new_rows
)

n_carried_labeled = int(
    template_df.iloc[
        :n_carried
    ][
        "manual_is_correct"
    ]
    .astype(str)
    .str.strip()
    .isin(
        [
            "0",
            "1",
        ]
    )
    .sum()
) if n_carried > 0 else 0

print()
print("===== ANNOTATION CARRY-FORWARD SUMMARY =====")
print("Previous rows:", previous_row_count)
print("Previous labeled rows:", previous_labeled_count)
print("Carried rows:", n_carried)
print("Carried labeled rows:", n_carried_labeled)
print("Changed rows requiring re-annotation:", changed_previous_count)
print("Previous rows absent from current dataset:", dropped_previous_count)
print("New / re-annotation rows:", n_new)
print("Total current annotation rows:", len(template_df))

if n_new > 0:
    print(
        "New batch range:",
        f"{first_new_batch} - {template_df['annotation_batch'].astype(int).max()}",
    )

if ANNOTATION_COMPLETED_PATH.exists():
    print()
    print(
        "UYARI: Current-version manual_annotation_completed.csv zaten mevcut."
    )
    print(
        "Bu dosya otomatik olarak değiştirilmedi. "
        "Yeni work CSV'yi tamamladıktan sonra current completed CSV'yi yeniden export edin."
    )


Annotation work CSV oluşturuldu:
/content/drive/MyDrive/LLM_Confidence_Thesis/data/annotations/validated_v003/manual_annotation_work.csv
Internal mapping: /content/drive/MyDrive/LLM_Confidence_Thesis/data/annotations/validated_v003/manual_annotation_internal_map.jsonl


## 9. Annotation instructions dosyası ve protocol manifest

Protocol manifest carry-forward kaynağını, previous completed CSV hash'ini ve kaç satırın
taşındığını kaydeder.


In [10]:
instructions = f"""# Manual Annotation Instructions

Dataset: `{VALIDATED_DATASET_VERSION}`

Protocol: `{ANNOTATION_PROTOCOL_VERSION}`

Previous annotation dataset: `{PREVIOUS_ANNOTATION_DATASET_VERSION}`

## Carry-forward davranışı

- Önceki dataset sürümündeki `manual_annotation_completed.csv` otomatik yüklenir.
- Önceki annotation satırı current dataset'te aynı soru, aynı blind model code, aynı frozen model answer,
  aynı reference/alias/type/category/source ile bulunuyorsa manual annotation alanları korunur.
- Değişmiş veya yeni satırlar yeniden annotation için boş bırakılır.
- Yeni satırlar önceki annotation'ların altına eklenir ve yeni batch numaraları alır.

## Doldurulacak kolonlar

### manual_is_correct

- `1`: model cevabı soruyu olgusal olarak doğru yanıtlıyor
- `0`: model cevabı yanlış / unknown / çelişkili / yetersiz

### manual_error_type

`manual_is_correct = 1` ise boş bırakabilirsiniz.

`manual_is_correct = 0` ise aşağıdakilerden birini yazın:

- `WRONG_FACT`
- `UNKNOWN`
- `MIXED_OR_CONTRADICTORY`
- `INCOMPLETE_OR_TRUNCATED`
- `IRRELEVANT_OR_FORMAT`
- `OTHER`

### annotation_notes

Opsiyonel kısa not.

### needs_review

- `0`: karar kesin
- `1`: tekrar gözden geçirilecek

Final freeze öncesi tüm `needs_review` değerleri `0` olmalıdır.

## Önemli

- `model_code` yalnız M1/M2/M3 şeklinde anonimdir.
- Model-code mapping dosyasını annotation bitene kadar açmayın.
- Confidence / logprob / entropy / verification / self-consistency sonuçlarına bakarak etiket vermeyin.
- Frozen model answer metnini düzeltmeyin.
- Reference answer'ın anlamsal olarak eşdeğer açık varyantlarını kabul edebilirsiniz.
- Bir cevap hem doğru cevabı hem de çelişkili/unknown ifade içeriyorsa `0` verin.
"""

write_instructions = True

if ANNOTATION_INSTRUCTIONS_PATH.exists():
    existing_instructions = (
        ANNOTATION_INSTRUCTIONS_PATH
        .read_text(
            encoding="utf-8"
        )
    )

    if (
        existing_instructions
        ==
        instructions
    ):
        write_instructions = False
    else:
        if not REBUILD_CURRENT_ANNOTATION_WORK:
            raise RuntimeError(
                "Existing annotation instructions current carry-forward protocol ile uyuşmuyor."
            )

        backup_path = backup_file(
            ANNOTATION_INSTRUCTIONS_PATH
        )

        print(
            "Eski instructions backup alındı:",
            backup_path,
        )

if write_instructions:
    ANNOTATION_INSTRUCTIONS_PATH.write_text(
        instructions,
        encoding="utf-8",
    )

previous_completed_sha256 = None

if (
    PREVIOUS_ANNOTATION_DATASET_VERSION
    is not None
):
    previous_completed_sha256 = (
        file_sha256(
            PREVIOUS_ANNOTATION_COMPLETED_PATH
        )
    )

protocol_manifest = {
    "stage": (
        "07_manual_annotation_protocol"
    ),
    "protocol_version": (
        ANNOTATION_PROTOCOL_VERSION
    ),
    "dataset_version": (
        VALIDATED_DATASET_VERSION
    ),
    "dataset_sha256": (
        dataset_sha256
    ),
    "previous_annotation_dataset_version": (
        PREVIOUS_ANNOTATION_DATASET_VERSION
    ),
    "previous_annotation_completed_file": (
        str(
            PREVIOUS_ANNOTATION_COMPLETED_PATH
        )
        if
        PREVIOUS_ANNOTATION_COMPLETED_PATH
        is not None
        else
        None
    ),
    "previous_annotation_completed_sha256": (
        previous_completed_sha256
    ),
    "carry_forward_matching_fields": (
        VISIBLE_CORE_FIELDS
    ),
    "n_previous_annotation_rows": (
        previous_row_count
    ),
    "n_carried_annotation_rows": (
        n_carried
    ),
    "n_carried_labeled_rows": (
        n_carried_labeled
    ),
    "n_changed_rows_requiring_reannotation": (
        changed_previous_count
    ),
    "n_previous_rows_absent_from_current_dataset": (
        dropped_previous_count
    ),
    "n_new_or_reannotation_rows": (
        n_new
    ),
    "stage04_completion_sha256": (
        file_sha256(
            STAGE04_COMPLETION_PATH
        )
    ),
    "stage06_v2_completion_sha256": (
        file_sha256(
            STAGE06_COMPLETION_PATH
        )
    ),
    "n_questions": (
        stage04_manifest[
            "n_questions"
        ]
    ),
    "n_models": (
        stage04_manifest[
            "n_models"
        ]
    ),
    "n_annotation_rows": (
        len(
            template_df
        )
    ),
    "model_blinding": True,
    "confidence_features_exposed": False,
    "row_shuffle_seed": (
        ROW_SHUFFLE_SEED
    ),
    "model_code_seed": (
        MODEL_CODE_SEED
    ),
    "annotation_batch_size": (
        ANNOTATION_BATCH_SIZE
    ),
    "editable_columns": (
        EDITABLE_COLUMNS
    ),
    "protected_fingerprint_fields": (
        PROTECTED_FINGERPRINT_FIELDS
    ),
    "incorrect_error_types": sorted(
        ERROR_TYPES_INCORRECT
    ),
    "annotation_work_file": str(
        ANNOTATION_TEMPLATE_PATH
    ),
    "annotation_completed_file": str(
        ANNOTATION_COMPLETED_PATH
    ),
    "internal_map_file": str(
        ANNOTATION_INTERNAL_MAP_PATH
    ),
    "instructions_file": str(
        ANNOTATION_INSTRUCTIONS_PATH
    ),
    "created_at_utc": (
        utc_now_iso()
    ),
}

write_protocol_manifest = True

if ANNOTATION_PROTOCOL_MANIFEST_PATH.exists():
    with open(
        ANNOTATION_PROTOCOL_MANIFEST_PATH,
        "r",
        encoding="utf-8",
    ) as f:
        existing_protocol_manifest = json.load(f)

    protocol_identity_fields = [
        "protocol_version",
        "dataset_version",
        "dataset_sha256",
        "previous_annotation_dataset_version",
        "previous_annotation_completed_sha256",
        "n_carried_annotation_rows",
        "n_new_or_reannotation_rows",
        "n_annotation_rows",
    ]

    same_protocol = all(
        existing_protocol_manifest.get(
            field
        )
        ==
        protocol_manifest.get(
            field
        )
        for field
        in protocol_identity_fields
    )

    if same_protocol:
        write_protocol_manifest = False
    else:
        if not REBUILD_CURRENT_ANNOTATION_WORK:
            raise RuntimeError(
                "Existing annotation protocol manifest "
                "current carry-forward protocol ile uyuşmuyor."
            )

        backup_path = backup_file(
            ANNOTATION_PROTOCOL_MANIFEST_PATH
        )

        print(
            "Eski protocol manifest backup alındı:",
            backup_path,
        )

if write_protocol_manifest:
    with open(
        ANNOTATION_PROTOCOL_MANIFEST_PATH,
        "x",
        encoding="utf-8",
    ) as f:
        json.dump(
            protocol_manifest,
            f,
            ensure_ascii=False,
            indent=2,
        )

print(
    "Instructions:",
    ANNOTATION_INSTRUCTIONS_PATH
)

print(
    "Protocol:",
    ANNOTATION_PROTOCOL_MANIFEST_PATH
)


Instructions: /content/drive/MyDrive/LLM_Confidence_Thesis/data/annotations/validated_v003/ANNOTATION_INSTRUCTIONS.md
Protocol: /content/drive/MyDrive/LLM_Confidence_Thesis/data/annotations/validated_v003/annotation_protocol_manifest.json


## 10. Annotation dosyasını açmadan önce kısa kontrol

Bu hücre yalnız annotation work CSV'nin ilk satırlarını gösterir.

Confidence veya model identity gösterilmez.

In [11]:
work_df = pd.read_csv(
    ANNOTATION_TEMPLATE_PATH,
    dtype=str,
    keep_default_na=False,
)

display(
    work_df.head(
        12
    )
)

print(
    "Rows:",
    len(
        work_df
    )
)

print(
    "Batches:",
    sorted(
        work_df[
            "annotation_batch"
        ].unique()
    )
)

print(
    "Model codes:",
    sorted(
        work_df[
            "model_code"
        ].unique()
    )
)

,annotation_row_id,annotation_batch,model_code,question_id,question,model_answer,reference_answer,accepted_aliases,answer_type,category,verification_source,manual_is_correct,manual_error_type,annotation_notes,needs_review
0,ANN000001,1,M2,TRQA000191,Bilginin temel kaynağının deneyim olduğunu sav...,"Deneyim felsefesine ""realizm"" denir.",Empirizm,Empirizm | Deneycilik | Empiricism,SHORT_PHRASE,Felsefe,,,,,0
1,ANN000002,1,M3,TRQA000176,İngiliz alfabesinde kaç harf vardır?,26,26,26 | yirmi altı,NUMBER,Dil,,,,,0
2,ANN000003,1,M2,TRQA000101,Yetişkin bir insanın iskeletinde tipik olarak ...,206 kemik.,206,206 | iki yüz altı,NUMBER,Bilim,,,,,0
3,ANN000004,1,M2,TRQA000183,Bir hektar kaç metrekaredir?,10000 metrekare.,10000 m²,10000 m² | 10.000 m² | 10000 metrekare | 10.00...,NUMBER_WITH_UNIT,Matematik,,,,,0
4,ANN000005,1,M2,TRQA000108,Bir deniz mili tam olarak kaç metredir?,1852 metresi.,1852 m,1852 m | 1.852 m | 1852 metre,NUMBER_WITH_UNIT,Coğrafya,,,,,0
5,ANN000006,1,M1,TRQA000093,İstanbul'un Osmanlılar tarafından fethedildiği...,1453,1453,1453,DATE,Tarih,,,,,0
6,ANN000007,1,M3,TRQA000147,Avrupa Merkez Bankası'nın merkezi hangi şehird...,Frankfurt,Frankfurt,Frankfurt | Frankfurt am Main,ENTITY_LOCATION,Ekonomi,,,,,0
7,ANN000008,1,M3,TRQA000168,Dünyanın ilk yapışkanlı posta pulu olarak kabu...,1840,1840,1840,DATE,Genel Kültür,,,,,0
8,ANN000009,1,M3,TRQA000119,DNA'nın klasik yapısı çift sarmal olarak tanım...,Evet.,Evet,Evet | Doğru | Yes,BINARY,Bilim,,,,,0
9,ANN000010,1,M1,TRQA000096,Sputnik 1 hangi yıl uzaya fırlatıldı?,1957.,1957,1957,DATE,Uzay,,,,,0


Rows: 600
Batches: ['1', '2', '3', '4', '5']
Model codes: ['M1', 'M2', 'M3']


# ANNOTATION'I ŞİMDİ YAPIN

Yukarıdaki hücrenin yazdırdığı `ANNOTATION_TEMPLATE_PATH` dosyasını Google Sheets / Excel ile açın.

Dosya current dataset version klasöründedir:

```text
data/annotations/<VALIDATED_DATASET_VERSION>/manual_annotation_work.csv
```

Önceki completed annotation'lar dosyanın üst kısmında **dolu** gelmelidir.
Yalnız yeni/değişmiş batch satırlarının `manual_is_correct` alanları boş olmalıdır.

Yalnız şu kolonları değiştirin:

```text
manual_is_correct
manual_error_type
annotation_notes
needs_review
```

Diğer kolonları değiştirmeyin.

Annotation tamamlandığında CSV olarak dışa aktarın ve current version klasörüne:

```text
manual_annotation_completed.csv
```

adıyla koyun.

Aşağıdaki hücreler annotation bittikten sonra çalıştırılacaktır.


## 11. Annotation progress

In [ ]:
if not ANNOTATION_COMPLETED_PATH.exists():
    raise FileNotFoundError(
        "Doldurulmuş annotation CSV bulunamadı:\n"
        f"{ANNOTATION_COMPLETED_PATH}\n\n"
        "Google Sheets'ten CSV olarak dışa aktar ve bu isimle Drive'a koy."
    )

annotated_df = pd.read_csv(
    ANNOTATION_COMPLETED_PATH,
    dtype=str,
    keep_default_na=False,
)

manual_clean = (
    annotated_df[
        "manual_is_correct"
    ]
    .astype(str)
    .str.strip()
)

review_clean = (
    annotated_df[
        "needs_review"
    ]
    .astype(str)
    .str.strip()
)

n_total = len(
    annotated_df
)

n_labeled = int(
    manual_clean.isin(
        [
            "0",
            "1",
        ]
    ).sum()
)

n_review = int(
    (
        review_clean
        ==
        "1"
    ).sum()
)

progress = (
    n_labeled
    /
    n_total
)

print(
    f"Labeled: {n_labeled}/{n_total} "
    f"({progress:.1%})"
)

print(
    "Needs review:",
    n_review
)

progress_by_batch = (
    annotated_df
    .assign(
        _labeled=manual_clean.isin(
            [
                "0",
                "1",
            ]
        ).astype(int),
        _review=(
            review_clean
            ==
            "1"
        ).astype(int),
    )
    .groupby(
        "annotation_batch"
    )
    .agg(
        n=(
            "annotation_row_id",
            "count",
        ),
        labeled=(
            "_labeled",
            "sum",
        ),
        needs_review=(
            "_review",
            "sum",
        ),
    )
    .reset_index()
)

display(
    progress_by_batch
)

Labeled: 0/360 (0.0%)
Needs review: 0


,annotation_batch,n,labeled,needs_review
0,1,120,0,0
1,2,120,0,0
2,3,120,0,0


## 12. Protected-column integrity + label validation

In [ ]:
annotated_df = pd.read_csv(
    ANNOTATION_COMPLETED_PATH,
    dtype=str,
    keep_default_na=False,
)

internal_df = pd.DataFrame(
    read_jsonl(
        ANNOTATION_INTERNAL_MAP_PATH
    )
)

if len(
    annotated_df
) != len(
    internal_df
):
    raise RuntimeError(
        "Annotation CSV / internal mapping row count mismatch."
    )

if annotated_df[
    "annotation_row_id"
].duplicated().any():
    raise RuntimeError(
        "Duplicate annotation_row_id."
    )

if set(
    annotated_df[
        "annotation_row_id"
    ]
) != set(
    internal_df[
        "annotation_row_id"
    ]
):
    raise RuntimeError(
        "Annotation row ID set değişmiş."
    )

internal_lookup = (
    internal_df
    .set_index(
        "annotation_row_id"
    )
    .to_dict(
        orient="index"
    )
)

protected_errors = []

for _, row in (
    annotated_df
    .iterrows()
):
    protected_payload = {
        field: (
            row[
                field
            ]
        )
        for field in (
            PROTECTED_FINGERPRINT_FIELDS
        )
    }

    actual_fp = (
        sha256_object(
            protected_payload
        )
    )

    expected_fp = (
        internal_lookup[
            row[
                "annotation_row_id"
            ]
        ][
            "protected_row_fingerprint"
        ]
    )

    if (
        actual_fp
        !=
        expected_fp
    ):
        protected_errors.append(
            row[
                "annotation_row_id"
            ]
        )

if protected_errors:
    raise RuntimeError(
        "Protected annotation kolonları değişmiş. "
        f"İlk kayıtlar: {protected_errors[:10]}"
    )

print(
    "Protected-column integrity: PASS"
)

# ------------------------------------------------------------
# Binary label validation
# ------------------------------------------------------------
manual_clean = (
    annotated_df[
        "manual_is_correct"
    ]
    .astype(str)
    .str.strip()
)

invalid_manual = (
    ~manual_clean.isin(
        [
            "",
            "0",
            "1",
        ]
    )
)

if invalid_manual.any():
    display(
        annotated_df.loc[
            invalid_manual,
            [
                "annotation_row_id",
                "manual_is_correct",
            ],
        ].head(
            50
        )
    )

    raise RuntimeError(
        "manual_is_correct yalnız boş/0/1 olabilir."
    )

review_clean = (
    annotated_df[
        "needs_review"
    ]
    .astype(str)
    .str.strip()
)

invalid_review = (
    ~review_clean.isin(
        [
            "",
            "0",
            "1",
        ]
    )
)

if invalid_review.any():
    raise RuntimeError(
        "needs_review yalnız boş/0/1 olabilir."
    )

error_clean = (
    annotated_df[
        "manual_error_type"
    ]
    .astype(str)
    .str.strip()
    .str.upper()
)

bad_error_rows = []

for idx, row in (
    annotated_df
    .iterrows()
):
    label = str(
        row[
            "manual_is_correct"
        ]
    ).strip()

    error_type = str(
        row[
            "manual_error_type"
        ]
    ).strip().upper()

    if label == "0":
        if (
            error_type
            not in
            ERROR_TYPES_INCORRECT
        ):
            bad_error_rows.append(
                row[
                    "annotation_row_id"
                ]
            )

    elif label == "1":
        if (
            error_type
            not in {
                "",
                "CORRECT",
            }
        ):
            bad_error_rows.append(
                row[
                    "annotation_row_id"
                ]
            )

if bad_error_rows:
    print(
        "Error-type problemi olan satırlar:",
        bad_error_rows[:30],
    )

    raise RuntimeError(
        "manual_error_type kurallarını düzeltin."
    )

print(
    "Annotation field validation: PASS"
)

Protected-column integrity: PASS
Annotation field validation: PASS


## 13. Finalize readiness

In [ ]:
manual_clean = (
    annotated_df[
        "manual_is_correct"
    ]
    .astype(str)
    .str.strip()
)

review_clean = (
    annotated_df[
        "needs_review"
    ]
    .astype(str)
    .str.strip()
)

missing_labels_df = (
    annotated_df.loc[
        ~manual_clean.isin(
            [
                "0",
                "1",
            ]
        ),
        [
            "annotation_row_id",
            "annotation_batch",
            "question_id",
            "model_code",
            "question",
            "model_answer",
        ],
    ]
)

review_rows_df = (
    annotated_df.loc[
        review_clean
        ==
        "1",
        [
            "annotation_row_id",
            "annotation_batch",
            "question_id",
            "model_code",
            "question",
            "model_answer",
            "reference_answer",
            "annotation_notes",
        ],
    ]
)

print(
    "Missing labels:",
    len(
        missing_labels_df
    )
)

print(
    "Needs review:",
    len(
        review_rows_df
    )
)

if not missing_labels_df.empty:
    display(
        missing_labels_df.head(
            50
        )
    )

if not review_rows_df.empty:
    display(
        review_rows_df.head(
            50
        )
    )

READY_TO_FINALIZE = (
    len(
        missing_labels_df
    )
    ==
    0
    and
    len(
        review_rows_df
    )
    ==
    0
)

print(
    "READY_TO_FINALIZE:",
    READY_TO_FINALIZE
)

Missing labels: 0
Needs review: 0
READY_TO_FINALIZE: True


## 14. Final labels üret ve Stage-07'yi dondur

Bu hücre yalnız `READY_TO_FINALIZE = True` olduğunda çalışır.

Final target dosyası yalnız model-answer key + manual labels içerir.
Confidence feature'ları buraya eklenmez.

In [ ]:
if not READY_TO_FINALIZE:
    raise RuntimeError(
        "Annotation henüz tamamlanmadı veya review kayıtları var."
    )

final_rows = []

for _, row in (
    annotated_df
    .iterrows()
):
    annotation_row_id = (
        row[
            "annotation_row_id"
        ]
    )

    internal = (
        internal_lookup[
            annotation_row_id
        ]
    )

    manual_label = int(
        str(
            row[
                "manual_is_correct"
            ]
        ).strip()
    )

    raw_error = (
        str(
            row[
                "manual_error_type"
            ]
        )
        .strip()
        .upper()
    )

    final_error = (
        "CORRECT"
        if manual_label
        ==
        1
        else raw_error
    )

    final_rows.append({
        "dataset_version": (
            VALIDATED_DATASET_VERSION
        ),
        "annotation_protocol_version": (
            ANNOTATION_PROTOCOL_VERSION
        ),
        "annotation_row_id": (
            annotation_row_id
        ),
        "question_id": (
            internal[
                "question_id"
            ]
        ),
        "model_id": (
            internal[
                "model_id"
            ]
        ),
        "model_revision": (
            internal[
                "model_revision"
            ]
        ),
        "stage04_inference_cache_key": (
            internal[
                "stage04_inference_cache_key"
            ]
        ),
        "model_answer_sha256": (
            internal[
                "model_answer_sha256"
            ]
        ),
        "manual_is_correct": (
            manual_label
        ),
        "manual_error_type": (
            final_error
        ),
        "annotation_notes": (
            str(
                row[
                    "annotation_notes"
                ]
            ).strip()
        ),
    })

if FINAL_LABELS_PATH.exists():
    raise FileExistsError(
        "Final labels zaten mevcut ve immutable kabul edilir:\n"
        f"{FINAL_LABELS_PATH}"
    )

write_jsonl_exclusive(
    FINAL_LABELS_PATH,
    final_rows,
)

pd.DataFrame(
    final_rows
).to_csv(
    FINAL_LABELS_CSV_PATH,
    index=False,
    encoding="utf-8-sig",
)

final_df = pd.DataFrame(
    final_rows
)

if len(
    final_df
) != expected_rows:
    raise RuntimeError(
        "Final manual labels coverage eksik."
    )

if final_df.duplicated(
    subset=[
        "question_id",
        "model_id",
    ]
).any():
    raise RuntimeError(
        "Final labels duplicate question×model."
    )

label_summary = (
    final_df
    .groupby(
        "model_id"
    )
    .agg(
        n=(
            "question_id",
            "count",
        ),
        n_correct=(
            "manual_is_correct",
            "sum",
        ),
        accuracy=(
            "manual_is_correct",
            "mean",
        ),
    )
    .reset_index()
)

display(
    label_summary
)

error_summary = (
    final_df.loc[
        final_df[
            "manual_is_correct"
        ]
        ==
        0
    ]
    .groupby(
        [
            "model_id",
            "manual_error_type",
        ]
    )
    .size()
    .reset_index(
        name="n"
    )
)

display(
    error_summary
)

LABEL_SUMMARY_PATH = (
    RESULTS_STAGE07_DIR /
    f"{VALIDATED_DATASET_VERSION}_label_summary.csv"
)

ERROR_SUMMARY_PATH = (
    RESULTS_STAGE07_DIR /
    f"{VALIDATED_DATASET_VERSION}_error_summary.csv"
)

label_summary.to_csv(
    LABEL_SUMMARY_PATH,
    index=False,
)

error_summary.to_csv(
    ERROR_SUMMARY_PATH,
    index=False,
)

completion = {
    "stage": (
        "07_manual_annotation"
    ),
    "dataset_version": (
        VALIDATED_DATASET_VERSION
    ),
    "annotation_protocol_version": (
        ANNOTATION_PROTOCOL_VERSION
    ),
    "previous_annotation_dataset_version": (
        PREVIOUS_ANNOTATION_DATASET_VERSION
    ),
    "n_carried_annotation_rows": (
        n_carried
    ),
    "n_new_or_reannotation_rows": (
        n_new
    ),
    "annotation_protocol_manifest_sha256": (
        file_sha256(
            ANNOTATION_PROTOCOL_MANIFEST_PATH
        )
    ),
    "dataset_sha256": (
        dataset_sha256
    ),
    "stage04_completion_sha256": (
        file_sha256(
            STAGE04_COMPLETION_PATH
        )
    ),
    "stage06_v2_completion_sha256": (
        file_sha256(
            STAGE06_COMPLETION_PATH
        )
    ),
    "n_questions": (
        stage04_manifest[
            "n_questions"
        ]
    ),
    "n_models": (
        stage04_manifest[
            "n_models"
        ]
    ),
    "n_model_answer_labels": (
        len(
            final_df
        )
    ),
    "all_labels_binary": True,
    "unresolved_review_count": 0,
    "confidence_features_exposed_during_annotation": False,
    "model_identity_blinded": True,
    "final_labels_file": str(
        FINAL_LABELS_PATH
    ),
    "final_labels_sha256": (
        file_sha256(
            FINAL_LABELS_PATH
        )
    ),
    "final_labels_csv": str(
        FINAL_LABELS_CSV_PATH
    ),
    "label_summary_file": str(
        LABEL_SUMMARY_PATH
    ),
    "error_summary_file": str(
        ERROR_SUMMARY_PATH
    ),
    "created_at_utc": (
        utc_now_iso()
    ),
}

if STAGE07_COMPLETION_PATH.exists():
    raise FileExistsError(
        "Stage-07 completion zaten mevcut:\n"
        f"{STAGE07_COMPLETION_PATH}"
    )

with open(
    STAGE07_COMPLETION_PATH,
    "x",
    encoding="utf-8",
) as f:
    json.dump(
        completion,
        f,
        ensure_ascii=False,
        indent=2,
    )

print(
    "STAGE 07 DONDURULDU:"
)

print(
    STAGE07_COMPLETION_PATH
)

print(
    "Final labels:"
)

print(
    FINAL_LABELS_PATH
)


,model_id,n,n_correct,accuracy
0,Qwen/Qwen3-4B,120,82,0.683333
1,google/gemma-3-4b-it,120,106,0.883333
2,microsoft/Phi-4-mini-instruct,120,57,0.475000


,model_id,manual_error_type,n
0,Qwen/Qwen3-4B,INCOMPLETE_OR_TRUNCATED,1
1,Qwen/Qwen3-4B,OTHER,1
2,Qwen/Qwen3-4B,UNKNOWN,5
3,Qwen/Qwen3-4B,WRONG_FACT,31
4,google/gemma-3-4b-it,OTHER,1
5,google/gemma-3-4b-it,UNKNOWN,1
6,google/gemma-3-4b-it,WRONG_FACT,12
7,microsoft/Phi-4-mini-instruct,IRRELEVANT_OR_FORMAT,1
8,microsoft/Phi-4-mini-instruct,MIXED_OR_CONTRADICTORY,6
9,microsoft/Phi-4-mini-instruct,UNKNOWN,21


STAGE 07 DONDURULDU:
/content/drive/MyDrive/LLM_Confidence_Thesis/results/stage07/validated_v002_stage07_complete.json
Final labels:
/content/drive/MyDrive/LLM_Confidence_Thesis/data/annotations/validated_v002/manual_labels_final.jsonl


# 07 tamamlanma kriterleri

- [ ] Validated dataset SHA doğrulandı
- [ ] Stage-04 frozen answers SHA doğrulandı
- [ ] Stage-06 V2 completion doğrulandı
- [ ] Previous annotation dataset version doğru seçildi
- [ ] Önceki completed annotation satırları değişmemiş item'lar için carry-forward edildi
- [ ] Yeni/değişmiş item'lar önceki annotation'ların altına yeni batch olarak eklendi
- [ ] Model identity M1/M2/M3 ile blind edildi
- [ ] Confidence / logprob / verification / consistency feature'ları annotation dosyasına konmadı
- [ ] Protected kolonlar fingerprint ile kilitlendi
- [ ] Current Stage-04 içindeki tüm model-answer label'ları tamamlandı
- [ ] Tüm `manual_is_correct` değerleri 0/1
- [ ] Yanlış cevaplarda error type girildi
- [ ] `needs_review = 1` kalan kayıt yok
- [ ] Final labels immutable JSONL olarak yazıldı
- [ ] Stage-07 completion manifest oluşturuldu

## Sonraki aşama — 08 Feature Assembly

08'de ilk kez:

```text
04 internal features
+
05 self-confidence / verification
+
06 V2 self-consistency
+
07 manual_is_correct
```

aynı leakage-safe master feature table'da birleştirilecek.
